In [1]:
%load_ext autoreload
%autoreload 2

import concurrent.futures
from cmipper import utils, config, parallelised_download_and_process, file_ops
import xarray as xa
import datetime
import numpy as np
from tqdm.auto import tqdm
from coralshift.plotting import spatial_plots

import dask.array as da
from dask import delayed, compute

from pathlib import Path

/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Extracting seafloor (duplicate run_main)

In [24]:
fps = Path("/maps/rt582/cmipper/.esgpull/data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/vo/gn/v20191216").glob("*.nc")
combined = xa.open_mfdataset(fps, chunks={"lev": -1, "i": -1, "j": -1})["vo"]   # levels as single chunk since will need to index along this dimension
combined["time"] = [datetime.datetime(vv.year,vv.month,vv.day) for vv in combined["time"].values]   # from 360 calendar to datetime
combined

<xarray.DataArray 'vo' (time: 1980, lev: 75, j: 1205, i: 1440)> Size: 1TB
dask.array<concatenate, shape=(1980, 75, 1205, 1440), dtype=float32, chunksize=(1, 75, 1205, 1440), chunktype=numpy.ndarray>
Coordinates:
  * lev        (lev) float64 600B 0.5058 1.556 2.668 ... 5.698e+03 5.902e+03
  * j          (j) int32 5kB 0 1 2 3 4 5 6 ... 1199 1200 1201 1202 1203 1204
  * i          (i) int32 6kB 0 1 2 3 4 5 6 ... 1434 1435 1436 1437 1438 1439
    latitude   (j, i) float32 7MB dask.array<chunksize=(1205, 1440), meta=np.ndarray>
    longitude  (j, i) float32 7MB dask.array<chunksize=(1205, 1440), meta=np.ndarray>
  * time       (time) datetime64[ns] 16kB 1850-01-16 1850-02-16 ... 2014-12-16
Attributes:
    standard_name:  sea_water_y_velocity
    long_name:      Sea Water Y Velocity
    comment:        mo: This variable is reported using a z* coordinate syste...
    units:          m s-1
    original_name:  mo: mask_copy((variable_name: vo), (variable_name: mask_3...
    cell_methods:   time: mean

In [13]:
def gen_seafloor_indices(xa_da: xa.DataArray, dim: str = "lev"):
    """Generate indices of seafloor values for a given variable in an xarray dataset.

    Args:
        xa_d (xa.Dataset): xarray dataset containing variable of interest
        var (str): name of variable of interest
        dim (str, optional): dimension along which to search for seafloor values. Defaults to "lev".

    Returns:
        indices_array (np.ndarray): array of indices of seafloor values for given variable
    """
    print("\ndetermining seafloor indices... ", flush=True)
    nans = np.isnan(xa_da).sum(dim=dim)  # separate out
    indices_array = -(nans.values) - 1
    indices_array[indices_array == -(len(xa_da[dim].values) + 1)] = -1
    return indices_array


def seafloor_inds(da):
    seafloor_indices = gen_seafloor_indices(da.isel(time=0))
    proj_seafloor_indices = np.broadcast_to(
                seafloor_indices,
                (
                    len(da.time),
                    len(da.j),
                    len(da.i),
                ),  # these variable names may differ by model
            )
    return proj_seafloor_indices

seafloor_indices = seafloor_inds(combined.isel(time=slice(0,4)))


determining seafloor indices... 


In [ ]:
combined.isel(time=slice(0,4)).chunks[:3]

In [5]:
lev_indices_da = da.from_array(seafloor_indices)
lev_indices_da

dask.array<array, shape=(4, 1205, 1440), dtype=int64, chunksize=(4, 1205, 1440), chunktype=numpy.ndarray>

In [7]:
from tqdm.auto import tqdm
# Extract the lev dimension size
lev_dim_size = len(combined.lev)

# Create a function to perform the extraction
def extract_lev_indexed(data, lev_indices, lev_dim_size):
    # Create an array of the same shape as lev_indices with an extra dimension for lev
    expanded_lev_indices = da.zeros((lev_dim_size, *lev_indices.shape), dtype=int)
    for lev in tqdm(range(lev_dim_size)):
        expanded_lev_indices[lev, ...] = (lev_indices == lev).astype(int)

    # Convert expanded_lev_indices to a DataArray
    expanded_lev_indices = xa.DataArray(expanded_lev_indices, dims=['lev', 'time', 'j', 'i'])

    # Perform the selection using where
    extracted_data = data.where(expanded_lev_indices, drop=True).sum('lev')
    
    return extracted_data

indexed_data = extract_lev_indexed(combined.isel(time=slice(0,4)), lev_indices_da, lev_dim_size)


100%|██████████| 75/75 [00:03<00:00, 18.79it/s]


KeyboardInterrupt: 

In [ ]:

def extract_lev(data, lev_indices):
    """
    Extract the data at specified lev indices.

    Parameters:
    - data: xarray DataArray of the NetCDF data.
    - lev_indices: Dask array with the lev indices.

    Returns:
    - Indexed data as a Dask array.
    """
    # Create a meshgrid of indices for time, i, and j
    time_idx, i_idx, j_idx = da.meshgrid(
        np.arange(data.shape[0]), 
        np.arange(data.shape[2]), 
        np.arange(data.shape[3]), 
        indexing='ij'
    )
    
    # Use advanced indexing to extract the desired values
    indexed_data = data[
        time_idx, 
        lev_indices_da, 
        i_idx, 
        j_idx
    ]
    
    return indexed_data

# Apply the function to extract the indexed data
indexed_data = extract_lev(combined, lev_indices_da)

# Convert the result to an xarray DataArray
result = xa.DataArray(indexed_data, dims=('time', 'i', 'j'))

In [ ]:
import xarray as xr
import numpy as np
import dask.array as da

# Load the NetCDF file
file_path = 'path_to_your_large_file.nc'
ds = xr.open_dataset(file_path, chunks={'time': 1})

# Assuming var is the variable you're interested in
var = 'your_variable_name'
data_var = ds[var]

# Example lev_indices numpy array of shape (time, i, j)
# Replace this with your actual lev_indices array
lev_indices = np.random.randint(0, len(ds.lev), size=(len(ds.time), len(ds.i), len(ds.j)))

# Convert lev_indices to a Dask array for efficient handling
lev_indices_da = da.from_array(lev_indices, chunks=data_var.chunks[:3])

# Create a new Dask array to hold the indexed data
def extract_lev(data, lev_indices):
    """
    Extract the data at specified lev indices.

    Parameters:
    - data: xarray DataArray of the NetCDF data.
    - lev_indices: Dask array with the lev indices.

    Returns:
    - Indexed data as a Dask array.
    """
    # Create a meshgrid of indices for time, i, and j
    time_idx, i_idx, j_idx = da.meshgrid(
        np.arange(data.shape[0]), 
        np.arange(data.shape[2]), 
        np.arange(data.shape[3]), 
        indexing='ij'
    )
    
    # Use advanced indexing to extract the desired values
    indexed_data = data[
        time_idx, 
        lev_indices_da, 
        i_idx, 
        j_idx
    ]
    
    return indexed_data

# Apply the function to extract the indexed data
indexed_data = extract_lev(data_var, lev_indices_da)

# Convert the result to an xarray DataArray
result = xr.DataArray(indexed_data, dims=('time', 'i', 'j'))

# Trigger the computation (optional, depends on your use case)
result.load()  # or result.compute() if you want to compute immediately

# Save the result to a new NetCDF file (optional)
result.to_netcdf('path_to_save_indexed_data.nc')


In [12]:
def seafloor(ds, var):
    seafloor_indices = utils.gen_seafloor_indices(ds.isel(time=0), var)
    proj_seafloor_indices = np.broadcast_to(
                seafloor_indices,
                (
                    len(ds.time),
                    len(ds.j),
                    len(ds.i),
                ),  # these variable names may differ by model
            )
    return extract_seafloor_indices(ds, proj_seafloor_indices, var)


def extract_seafloor_indices(ds, seafloor_indices, variable_id):
    new_ds = ds.copy()
    
    # Ensure seafloor_indices is computed before using it
    seafloor_indices = compute(seafloor_indices)[0]
    
    # Parallelize the extraction
    cmip6_array = delayed(utils.extract_3d_index_vals)(new_ds[variable_id], seafloor_indices)
    
    # Convert the delayed array to a Dask array
    cmip6_array = da.from_delayed(cmip6_array, shape=(len(ds.time), len(ds.j), len(ds.i)), dtype=new_ds[variable_id].dtype)
    
    new_ds[variable_id] = (["time", "j", "i"], cmip6_array)
    
    return new_ds

In [34]:
def fp_extract(fp):
    ds = xa.open_dataset(fp, chunks={"lev": -1, "i": -1, "j": -1})
    return seafloor(ds, "vo")

In [35]:
from dask.distributed import Client
# client = Client(n_workers=4, threads_per_worker=8, memory_limit='8GB')
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:42399/status,
Dashboard: http://127.0.0.1:42399/status,Workers: 16
Total threads: 256,Total memory: 0.98 TiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:44597,Workers: 16
Dashboard: http://127.0.0.1:42399/status,Total threads: 256
Started: Just now,Total memory: 0.98 TiB
Comm: tcp://127.0.0.1:40631,Total threads: 16
Dashboard: http://127.0.0.1:45471/status,Memory: 62.98 GiB
Nanny: tcp://127.0.0.1:33805,


In [36]:
extract_parallel = delayed(fp_extract)
fps = [f for f in Path("/maps/rt582/cmipper/.esgpull/data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/vo/gn/v20191216").iterdir() if f.is_file()]
tasks = [fp_extract(fp) for fp in list(fps)[:6]]


determining seafloor indices... 

determining seafloor indices... 

determining seafloor indices... 

determining seafloor indices... 

determining seafloor indices... 

determining seafloor indices... 


In [37]:
results = da.compute(*tasks)

2024-06-19 12:00:46,187 - distributed.protocol.core - CRITICAL - Failed to Serialize
Traceback (most recent call last):
  File "/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/site-packages/distributed/protocol/core.py", line 109, in dumps
    frames[0] = msgpack.dumps(msg, default=_encode_default, use_bin_type=True)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/maps/rt582/miniforge3/envs/shiftpy/lib/python3.12/site-packages/msgpack/__init__.py", line 36, in packb
    return Packer(**kwargs).pack(o)
           ^^^^^^^^^^^^^^^^^^^^^^^^
  File "msgpack/_packer.pyx", line 294, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 300, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 297, in msgpack._cmsgpack.Packer.pack
  File "msgpack/_packer.pyx", line 264, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 231, in msgpack._cmsgpack.Packer._pack
  File "msgpack/_packer.pyx", line 264, in msgp

CancelledError: ('open_dataset-time_bnds-9dbee626718b83d7405f91f5f0227921', 119, 0)

In [ ]:
%timeit computed_result = result.compute()  # 6.18 s ± 2.73 s per loop for t=4, -1 lev chunking
# 1min 12s ± 18.7 s per loop for t=16!

In [ ]:
combined["vo"]

In [ ]:
out = xa.open_dataset("https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/rsdo/gn/v20191216/rsdo_Omon_HadGEM3-GC31-MM_historical_r1i1p1f3_gn_201001-201412.nc")
out

In [ ]:
xa.open_dataset(
    "https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/rsdo/gn/v20191216/rsdo_Omon_HadGEM3-GC31-MM_historical_r1i1p1f3_gn_201001-201412.nc",
    chunks={"time": 5, "i": 90, "j": 90}
    ).isel(lev=slice(0,20)).load()

In [ ]:
from tqdm.auto import tqdm
arrs = [
    # [48, 180, 180], # 2939s (49m)
    # [5, 180, 180],  # 1677s (28m)
    # [10, 180, 180], # 1824s (30m)
    # [5, 360, 360],  # 1465s (24m)
    # [1, 180, 180],  # 1921s (32m)
    [10, 360, 360], # 4745 (1h 19m)
    [10, 480, 480], 
    # [10, 180, 180],
    # [5, 360, 360], 
    # [1, 180, 180], 
]

times = np.zeros(len(arrs))
import time
for arr_i, arr in tqdm(enumerate(arrs)):
    try:
        start = time.time()
        out = xa.open_dataset(
            "https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/rsdo/gn/v20191216/rsdo_Omon_HadGEM3-GC31-MM_historical_r1i1p1f3_gn_201001-201412.nc#log",
            chunks={"time": arr[0], "i": arr[1], "j": arr[2]}
            ).isel(lev=slice(0,20)).load()
        dur = time.time() - start
        times[arr_i] = dur
    except RuntimeError:
        print("failed:", arr)
     
        

In [ ]:
from tqdm.auto import tqdm
arrs = [
    # [48, 180, 180], # 2939s (49m)
    # [5, 180, 180],  # 1677s (28m)
    # [10, 180, 180], # 1824s (30m)
    # [5, 360, 360],  # 1465s (24m)
    # [1, 180, 180],  # 1921s (32m)
    # [10, 360, 360], # 4745 (1h 19m)
    [20, 3], # more than an hour
    # [10, 180, 180],
    # [5, 360, 360], 
    # [1, 180, 180], 
]

times = np.zeros(len(arrs))
import time
for arr_i, arr in tqdm(enumerate(arrs)):
    try:
        start = time.time()
        out = xa.open_dataset(
            "https://esgf-data1.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/MOHC/HadGEM3-GC31-MM/historical/r1i1p1f3/Omon/rsdo/gn/v20191216/rsdo_Omon_HadGEM3-GC31-MM_historical_r1i1p1f3_gn_201001-201412.nc#log",
            chunks={"lev": arr[0], "time": arr[1]}
            ).isel(lev=slice(0,20)).load()
        dur = time.time() - start
        times[arr_i] = dur
    except RuntimeError:
        print("failed:", arr)
     
        

In [ ]:
xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/HadGEM3-GC31-MM/r1i1p1f3/og_grid/vo/vo_uncropped_sfl-20_tp_201001-201412.nc").isel(lev=0,time=0)["vo"].plot()

In [ ]:
xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/HadGEM3-GC31-MM/r1i1p1f3/regridded/uo/uo_uncropped_sfl-20_ll_198801-198912.nc")

In [ ]:
1465/60

In [ ]:
np.diff(times)

In [ ]:
fps = Path("/maps/rt582/cmipper/data/env_vars/cmip6/HadGEM3-GC31-MM/r1i1p1f3/regridded/uo/").glob("*.nc")
ds = xa.open_mfdataset(fps)

In [ ]:
ds

In [ ]:
ds.isel(time=0).uo.plot()

In [ ]:
ds = xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/HadGEM3-GC31-MM/r1i1p1f3/regridded/wfo/wfo_uncropped_sfc_ll_195001-196912.nc")
ds.isel(time=0)["wfo"].plot()

In [ ]:
import numpy as np
np.nanmax(ds.isel(time=0)["wfo"].values)

In [ ]:
regridded = utils.cdo_remap(ds, "/maps/rt582/cmipper/data/env_vars/cmip6/HadGEM3-GC31-MM/HadGEM3-GC31-MM_remap_template.txt")
regridded

In [ ]:
regridded.isel(time=0)["hfds"].plot()

In [ ]:
#!/usr/bin/env python
from __future__ import print_function
import requests
import xml.etree.ElementTree as ET
import numpy

# Author: Unknown
# I got the original version from a word document published by ESGF
# https://docs.google.com/document/d/1pxz1Kd3JHfFp8vR2JCVBfApbsHmbUQQstifhGNdc6U0/edit?usp=sharing

# API AT: https://github.com/ESGF/esgf.github.io/wiki/ESGF_Search_REST_API#results-pagination

def esgf_search(server="https://esgf-node.llnl.gov/esg-search/search",
                files_type="OPENDAP", local_node=True, project="CMIP6",
                verbose=False, format="application%2Fsolr%2Bjson",
                use_csrf=False, **search):
    client = requests.session()
    payload = search
    payload["project"] = project
    payload["type"]= "File"
    if local_node:
        payload["distrib"] = "false"
    if use_csrf:
        client.get(server)
        if 'csrftoken' in client.cookies:
            # Django 1.6 and up
            csrftoken = client.cookies['csrftoken']
        else:
            # older versions
            csrftoken = client.cookies['csrf']
        payload["csrfmiddlewaretoken"] = csrftoken

    payload["format"] = format

    offset = 0
    numFound = 10000
    all_files = []
    files_type = files_type.upper()
    while offset < numFound:
        payload["offset"] = offset
        url_keys = [] 
        for k in payload:
            url_keys += ["{}={}".format(k, payload[k])]

        url = "{}/?{}".format(server, "&".join(url_keys))
        # print(url)
        r = client.get(url)
        r.raise_for_status()
        resp = r.json()["response"]
        numFound = int(resp["numFound"])
        resp = resp["docs"]
        offset += len(resp)
        for d in resp:
            if verbose:
                for k in d:
                    print("{}: {}".format(k,d[k]))
            url = d["url"]
            for f in d["url"]:
                sp = f.split("|")
                if sp[-1] == files_type:
                    all_files.append(sp[0].split(".html")[0])
    return sorted(all_files)

In [ ]:
esgf_search(activity_id='CMIP', table_id='Omon', experiment_id='historical',
                  institution_id="AWI", source_id="AWI-CM-1-1-MR")

In [ ]:
parallelised_download_and_process.main(source_id="MPI-ESM1-2-HR", variable_id="po4")

In [ ]:
parallelised_download_and_process.generate_bash_commands()

In [ ]:
file_ops.read_yaml(config.model_info).keys()

In [ ]:
parallelised_download_and_process.main(source_id="MPI-ESM1-2-HR", variable_id="no3")

In [ ]:
from pyesgf.search import SearchConnection
conn = SearchConnection('https://esgf-data.dkrz.de/esg-search', distrib=True)
ctx = conn.new_context(
    project='CMIP6',
    source_id='MPI-ESM1-2-HR',
    experiment_id='historical',
    variable='no3',
    frequency='mon',
    variant_label='r1i1p1f1',
    data_node='esgf-data1.llnl.gov')
ctx.hit_count
result = ctx.search()[0]


In [ ]:
result.dataset_id
files = result.file_context().search()
for file in tqdm(files):
    print(file.opendap_url)

In [ ]:
xa.open_dataset(files[0].opendap_url, chunks={'time': 120}).load()

In [ ]:
ctx.hit_count
# ctx.search()[0]

In [ ]:
xa.open_dataset("/maps-priv/maps/rt582/cmipper/data/env_vars/cmip6/MPI-ESM1-2-HR/r1i1p1f1/regridded/no3/no3_uncropped_sfl-20_ll_195001-195412.nc").isel(time=0)["no3"].plot()

In [ ]:
huge = utils.process_xa_d(xa.open_dataset("/maps/rt582/cmipper/temp_data/temp_huge.nc", chunks="auto"))
huge

In [ ]:
huge_by_time = utils.process_xa_d(xa.open_dataset("/maps/rt582/cmipper/temp_data/temp_huge.nc", chunks={"time": 10}))
huge_auto = utils.process_xa_d(xa.open_dataset("/maps/rt582/cmipper/temp_data/temp_huge.nc", chunks="auto"))


%timeit huge_by_time.mean(dim="time").compute()
%timeit huge_auto.mean(dim="time").compute()

In [ ]:
url = "http://aims3.llnl.gov/thredds/dodsC/css03_data/CMIP6/CMIP/NOAA-GFDL/GFDL-CM4/historical/r1i1p1f1/Omon/chl/gr/v20180701/chl_Omon_GFDL-CM4_historical_r1i1p1f1_gr_185001-186912.nc"
out = xa.open_dataset(url)

In [ ]:
xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/MPI-ESM1-2-HR/r1i1p1f1/og_grid/no3/no3_uncropped_sfl-20_tp_189001-189412.nc")

In [ ]:
out.isel(lev=-1, time=0)["chl"].plot()

In [ ]:
huge["no3"].mean().compute()

In [ ]:
og = xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/MPI-ESM1-2-HR/r1i1p1f1/og_grid/no3/no3_uncropped_sfl-20_tp_190001-190412.nc")
og

In [ ]:
whole_gebco = xa.open_dataset("/maps/rt582/cmipper/temp_data/GEBCO_2023.nc")
lim_gebco = whole_gebco.isel(lat=slice(0,-1), lon=slice(0, -1))

In [ ]:
utils.generate_remapping_file(
    lim_gebco, 
    "/maps/rt582/cmipper/temp_data/GEBCO_2023_remapper.txt",
    resolutions=[0.1,0.1]
    )

In [ ]:
lim_gebco.rio.resolution()

In [ ]:
remapped = utils.cdo_remap_fly(
    "/maps/rt582/cmipper/data/env_vars/cmip6/MPI-ESM1-2-HR/r1i1p1f1/regridded/no3/no3_uncropped_sfl-20_ll_190501-190912.nc",
    [0.01,0.01], "no3")

In [ ]:
out = utils.process_xa_d(remapped)

In [ ]:
out

In [ ]:
spatial_plots.plot_spatial(
    utils.process_xa_d(
    xa.open_dataset(
        "/maps/rt582/cmipper/data/env_vars/cmip6/MPI-ESM1-2-HR/r1i1p1f1/regridded/no3/no3_uncropped_sfl-20_ll_190501-190912.nc")).sel(
            latitude=slice(-20,-0), longitude=slice(140,150))["no3"].isel(time=0))

In [ ]:
spatial_plots.plot_spatial(out.sel(latitude=slice(-20,-0), longitude=slice(140,150))["no3"].isel(time=0))

In [ ]:
# remapped.to_netcdf("/maps/rt582/cmipper/temp_data/temp_huge.nc")

# To do
- Make dasking on download more efficient (to increase download speed)
- Practise dask functionality on large file (for subsetting and running statistics on mfdatasets of high-resolution files)
- Decide on data source (resolution vs. past-future availability). How could I take advantage of the daily tos? Use this instead monthly to calculate statistics? (Since would calculate on coarse then interpolate, this wouldn't be a huge performance issue)
- Download data
- Reconsider resampling schema: use cdo? 
- Dask client processsing of statistics
- Buy dive kit!

In [ ]:
spatial_plots.plot_spatial(utils.process_xa_d(remapped["no3"]).isel(time=0).sel(lat=slice(-40,0), lon=slice(130,170)))

In [1]:
from cdo import *

cdo = Cdo()

In [ ]:
cdo.remapbil("/maps/rt582/cmipper/temp_data/GEBCO_2023_remapper.txt", input=lim_gebco, returnXDataset=True)

In [ ]:
out = utils.cdo_remap(
    lim_gebco,
    "/maps/rt582/cmipper/temp_data/GEBCO_2023_remapper.txt",
    "elevation"
)["elevation"]

In [ ]:
out.to_netcdf("/maps/rt582/cmipper/temp_data/temp.nc")

In [ ]:
xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/EC-Earth3P-HR/r1i1p2f1/regridded/concatted_vars_N0_S-32_W130_E170/hfds_mlotst_rsdo_so_thetao_tos_umo_uo_vmo_vo_wfo_N0_S-32_W130_E170_levs_0-20_ll_195000-204912.nc")

In [ ]:
model_info_dict = utils.read_yaml(config.model_info)
download_config_dict = utils.read_yaml(config.download_config)
limited_dict = utils.limit_model_info_dict(model_info_dict, download_config_dict)
limited_dict

In [ ]:
from pathlib import Path

fps = Path("/maps/rt582/cmipper/data/env_vars/cmip6/EC-Earth3P-HR/r1i1p2f1/regridded/concatted_vars_N0_S-32_W130_E170").glob("*.nc")


file_ops.find_files_for_time(fps, [1950, 2060])

In [ ]:
source_id = 'EC-Earth3P-HR'
variables = list(limited_dict[source_id]["variable_dict"].keys())
member_id=limited_dict[source_id]["member_ids"][0]
lats = download_config_dict["lats"]
lons = download_config_dict["lons"]

In [ ]:
# finding existing files
file_ops.find_intersecting_cmip(variables=variables, source_id=source_id, member_id=limited_dict[source_id]["member_ids"][0], lats=lats, lons=lons, year_range=(1950, 2040))

In [ ]:
source_ids = list(limited_dict.keys())
for source_id in source_ids:
    print(f"Processing {source_id}")
    for member_id in limited_dict[source_id]["member_ids"]:
        for experiment_id in limited_dict[source_id]["experiment_ids"]:
            for variable_id in limited_dict[source_id]["variable_dict"].keys():
                print(member_id, experiment_id, variable_id)
                parallelised_download_and_process.concat_cmip_files_by_time(source_id=source_id, experiment_id=experiment_id, member_id=member_id, variable_id=variable_id)
            # once all concatted by time and ready for merging
            parallelised_download_and_process.merge_cmip_data_by_variables(source_id=source_id, experiment_id=experiment_id, member_id=member_id)     

In [ ]:
xa.open_dataset("/maps/rt582/cmipper/data/env_vars/cmip6/EC-Earth3P-HR/r1i1p2f1/newtest/regridded/concatted_vars_N0_S-32_W130_E170/rsdo_N0_S-32_W130_E170_sfl-20_ll_195000-201412.nc")

In [ ]:
found = file_ops.find_intersecting_cmip(
    variables = ["tos"],
    lats = [-10,0]
)

In [ ]:
found[1]

In [ ]:
# MVP, from https://stackoverflow.com/questions/44989473/nesting-concurrent-futures-threadpoolexecutor

def inner(i, j):
    return i, j, i**j


def outer(i):
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(inner, i, j): j for j in range(5)}
        results = []
        for future in concurrent.futures.as_completed(futures):
            results.append(future.result())
    return results


def main():
    with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
        futures = {executor.submit(outer, i): i for i in range(10)}
        results = []
        for future in concurrent.futures.as_completed(futures):
            results.extend(future.result())
    print(results)


if __name__ == "__main__":
    main()

In [ ]:
model_info_dict = utils.read_yaml(config.model_info)
download_config_dict = utils.read_yaml(config.download_config)
utils.limit_model_info_dict(model_info_dict, download_config_dict)['EC-Earth3P-HR']["variable_dict"].keys()


    

In [ ]:
import concurrent.futures
from cmipper import utils, config

def test_func(arg1, arg2, arg3):
    print(arg1, arg2, arg3)

def execute_functions_in_threadpool(args):
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=12) as executor:
            futures = [executor.submit(parallelised_download_and_process.download_cmip_variable_data, *arg) for arg in args]
            return futures
    except Exception as e:
        print(f"An error occurred: {e}")


def main():
    model_info_dict = utils.read_yaml(config.model_info)
    download_config_dict = utils.read_yaml(config.download_config)
    limited_download_dict = utils.limit_model_info_dict(model_info_dict, download_config_dict)
    
    source_ids = ["EC-Earth3P-HR"]
    member_ids = limited_download_dict[source_ids[0]]["member_ids"]
    variable_ids = limited_download_dict[source_ids[0]]["variable_dict"].keys()

    try:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = [executor.submit(execute_functions_in_threadpool, [(source_id, member_id, variable_id)]) 
                       for source_id in source_ids for member_id in member_ids for variable_id in variable_ids]

            # Wait for all futures to complete
            concurrent.futures.wait(futures)
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":
    main()


# WIP: Attempt at parallelised logging

In [ ]:
def main():
    model_info_dict = utils.read_yaml(config.model_info)
    download_config_dict = utils.read_yaml(config.download_config)

    source_ids = ["EC-Earth3P-HR"]
    member_ids = model_info_dict[source_ids[0]]["member_ids"]
    variable_ids = download_config_dict["variable_ids"]

    try:
        with concurrent.futures.ThreadPoolExecutor() as executor:
            futures = []
            for source_id in source_ids:
                for member_id in member_ids:
                    for variable_id in variable_ids:
                        log_fp = config.logging_dir / source_id / member_id / "_".join([variable_id, "download.log"])
                        print(log_fp)
                        if not log_fp.parent.exists():
                            log_fp.parent.mkdir(parents=True)
                        utils.redirect_stdout_stderr_to_file(log_fp)
                        # futures.extend(executor.submit(utils.execute_functions_in_threadpool, [(source_id, member_id, variable_id)]))
                        futures = [executor.submit(utils.execute_functions_in_threadpool, [(source_id, member_id, variable_id)]) 
                                for source_id in source_ids for member_id in member_ids for variable_id in variable_ids]

            # Wait for all futures to complete
            concurrent.futures.wait(futures)
    except Exception as e:
        print(f"An error occurred: {e}")

if __name__ == "__main__":

    main()